In [12]:
import os

from atlassian import Confluence
from notion_client import Client

from confluence_to_notion.converter.parser import parse_storage_xml
from confluence_to_notion.converter.builder import build_notion_blocks

In [2]:
CONFLUENCE_TEST_SPACE = "SPC"

In [3]:
# Confluence API client setup
url = os.environ["CONFLUENCE_DC_URL"]
pat = os.environ["CONFLUENCE_DC_PAT"]
confluence_client = Confluence(url=url, token=pat)

In [ ]:
notion_client = Client(auth=os.environ["NOTION_API_KEY"])

In [5]:
def create_confluence_page(title: str, storage_xml: str) -> str:
    result = confluence_client.create_page(
        space=CONFLUENCE_TEST_SPACE,
        title=title,
        body=storage_xml,
        representation="storage",
    )
    page_id =str(result["id"])
    return page_id

In [ ]:
title = "太字テキスト"
xml = "<p><strong>太字テキスト</strong></p>"

confluence_page_id = create_confluence_page(title, xml)

'2523137'

In [7]:
confluence_page_id = "2523137"
page = confluence_client.get_page_by_id(
    confluence_page_id, expand="body.storage,title"
)
title = page["title"]
storage_xml = page["body"]["storage"]["value"]

In [8]:
title

'太字テキスト'

In [9]:
storage_xml

'<p><strong>太字テキスト</strong></p>'

In [14]:
conversion_nodes = parse_storage_xml(storage_xml)

In [16]:
conversion_nodes[0]

ConversionNode(node_type='paragraph', rich_text=[{'type': 'text', 'text': {'content': '太字テキスト', 'link': None}, 'annotations': {'bold': True, 'italic': False, 'strikethrough': False, 'underline': False, 'code': False, 'color': 'default'}, 'plain_text': '太字テキスト'}], children=[], attrs={})

In [17]:
blocks = build_notion_blocks(conversion_nodes)

In [18]:
blocks

[{'object': 'block',
  'type': 'paragraph',
  'paragraph': {'rich_text': [{'type': 'text',
     'text': {'content': '太字テキスト', 'link': None},
     'annotations': {'bold': True,
      'italic': False,
      'strikethrough': False,
      'underline': False,
      'code': False,
      'color': 'default'},
     'plain_text': '太字テキスト'}]}}]

In [24]:
def create_notion_page(parent_page_id: str, title: str, blocks: list[dict]) -> str:
    """ページを作成し、ページIDを返す"""
    # Notion API は一度に100ブロックまでしか受け付けない
    first_batch = blocks[:100]
    rest = blocks[100:]

    page = notion_client.pages.create(
        parent={"page_id": parent_page_id},
        properties={
            "title": {"title": [{"text": {"content": title}}]}
        },
        children=first_batch,
    )
    page_id = page["id"]

    # 100ブロックを超える場合は追加
    for i in range(0, len(rest), 100):
        notion_client.blocks.children.append(
            block_id=page_id,
            children=rest[i:i + 100],
        )

    return page_id

In [27]:
notion_parent_page_id=os.environ["NOTION_TEST_PAGE_ID"]

notion_page_id = create_notion_page(
    parent_page_id=notion_parent_page_id,
    title=title,
    blocks=blocks,
)

In [28]:
notion_page_id

'36519e08-5037-8176-9521-ca8db81ee911'

In [29]:
def get_blocks(page_id: str) -> list[dict]:
    """ページのブロック一覧を全件取得する（has_more 対応）"""
    blocks = []
    cursor = None
    while True:
        kwargs = {"block_id": page_id}
        if cursor:
            kwargs["start_cursor"] = cursor
        response = notion_client.blocks.children.list(**kwargs)
        blocks.extend(response["results"])
        if not response["has_more"]:
            break
        cursor = response["next_cursor"]
    return blocks

In [34]:
blocks = get_blocks("36119e0850378000bacef4ffa55bceea")

In [35]:
blocks[0]

{'object': 'block',
 'id': '36519e08-5037-8037-bccb-f12f2588e3fe',
 'parent': {'type': 'page_id',
  'page_id': '36119e08-5037-8000-bace-f4ffa55bceea'},
 'created_time': '2026-05-19T08:34:00.000Z',
 'last_edited_time': '2026-05-19T08:34:00.000Z',
 'created_by': {'object': 'user',
  'id': 'e56262de-2e35-4e80-97cf-f8e5f62620a5'},
 'last_edited_by': {'object': 'user',
  'id': 'e56262de-2e35-4e80-97cf-f8e5f62620a5'},
 'has_children': True,
 'in_trash': False,
 'type': 'child_page',
 'child_page': {'title': 'Notionにようこそ！ (1)'},
 'archived': False}

In [37]:
blocks[1]

{'object': 'block',
 'id': '36119e08-5037-808a-9f9f-e54d1a82b313',
 'parent': {'type': 'page_id',
  'page_id': '36119e08-5037-8000-bace-f4ffa55bceea'},
 'created_time': '2026-05-15T07:42:00.000Z',
 'last_edited_time': '2026-05-15T07:42:00.000Z',
 'created_by': {'object': 'user',
  'id': 'e56262de-2e35-4e80-97cf-f8e5f62620a5'},
 'last_edited_by': {'object': 'user',
  'id': 'e56262de-2e35-4e80-97cf-f8e5f62620a5'},
 'has_children': False,
 'in_trash': False,
 'type': 'to_do',
 'to_do': {'rich_text': [{'type': 'text',
    'text': {'content': 'Notionアカウントを作成', 'link': None},
    'annotations': {'bold': False,
     'italic': False,
     'strikethrough': False,
     'underline': False,
     'code': False,
     'color': 'default'},
    'plain_text': 'Notionアカウントを作成',
    'href': None}],
  'checked': True,
  'color': 'default'},
 'archived': False}

In [42]:
blocks[2]

{'object': 'block',
 'id': '36119e08-5037-800d-8895-de7c2e5e9763',
 'parent': {'type': 'page_id',
  'page_id': '36119e08-5037-8000-bace-f4ffa55bceea'},
 'created_time': '2026-05-15T07:42:00.000Z',
 'last_edited_time': '2026-05-15T07:42:00.000Z',
 'created_by': {'object': 'user',
  'id': 'e56262de-2e35-4e80-97cf-f8e5f62620a5'},
 'last_edited_by': {'object': 'user',
  'id': 'e56262de-2e35-4e80-97cf-f8e5f62620a5'},
 'has_children': True,
 'in_trash': False,
 'type': 'to_do',
 'to_do': {'rich_text': [{'type': 'text',
    'text': {'content': 'このページの下で、', 'link': None},
    'annotations': {'bold': False,
     'italic': False,
     'strikethrough': False,
     'underline': False,
     'code': False,
     'color': 'default'},
    'plain_text': 'このページの下で、',
    'href': None},
   {'type': 'text',
    'text': {'content': ' ', 'link': None},
    'annotations': {'bold': True,
     'italic': False,
     'strikethrough': False,
     'underline': False,
     'code': False,
     'color': 'default'},
  

In [43]:
blocks[3]

{'object': 'block',
 'id': '36119e08-5037-809a-9d16-f24ba4f8d404',
 'parent': {'type': 'page_id',
  'page_id': '36119e08-5037-8000-bace-f4ffa55bceea'},
 'created_time': '2026-05-15T07:42:00.000Z',
 'last_edited_time': '2026-05-15T07:42:00.000Z',
 'created_by': {'object': 'user',
  'id': 'e56262de-2e35-4e80-97cf-f8e5f62620a5'},
 'last_edited_by': {'object': 'user',
  'id': 'e56262de-2e35-4e80-97cf-f8e5f62620a5'},
 'has_children': False,
 'in_trash': False,
 'type': 'to_do',
 'to_do': {'rich_text': [{'type': 'text',
    'text': {'content': '左側の再度バーを利用して、', 'link': None},
    'annotations': {'bold': False,
     'italic': False,
     'strikethrough': False,
     'underline': False,
     'code': False,
     'color': 'default'},
    'plain_text': '左側の再度バーを利用して、',
    'href': None},
   {'type': 'text',
    'text': {'content': '検索、管理、新ページの追加 ', 'link': None},
    'annotations': {'bold': True,
     'italic': False,
     'strikethrough': False,
     'underline': False,
     'code': False,
     '